# Primary-State PPO Full Training

This is the state-only PPO comparison for four profiles and five seeds, not a legacy attention/GRU experiment. Every full run starts fresh, test trajectories remain sealed, and full training is locked by default.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPOSITORY_DIR = Path('/content/VitalDB-Feature-Selection')
DRIVE_PROJECT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
RUN_FULL_TRAINING = False
FULL_CONFIRMATION = ''
RUN_ANALYSIS_ONLY = False

def run(command, cwd=None, env=None, stage=None, capture_output=False):
    command = list(map(str, command))
    print('$', ' '.join(command), flush=True)
    completed = subprocess.run(command, cwd=cwd, env=env, check=False, text=capture_output, capture_output=capture_output)
    if capture_output and completed.stdout:
        print(completed.stdout, end='')
    if capture_output and completed.stderr:
        print(completed.stderr, end='', file=sys.stderr)
    if completed.returncode != 0:
        path_context = ''
        if env is not None:
            names = ('VITALDB_MODELING_DATASET_DIR', 'VITALDB_DEMOGRAPHICS_CSV', 'VITALDB_PROJECT_DATA_ROOT', 'VITALDB_EXPECTED_COHORT_FINGERPRINT')
            path_context = '\n'.join(name + '=' + env.get(name, '<unset>') for name in names)
        stdout = (completed.stdout or '')[-20000:] if capture_output else '<streamed above>'
        stderr = (completed.stderr or '')[-20000:] if capture_output else '<streamed above>'
        stage_name = stage or 'Command'
        raise RuntimeError(f'{stage_name} failed.\ncommand={command}\nexit_code={completed.returncode}\ncwd={cwd}\n{path_context}\nstdout:\n{stdout}\nstderr:\n{stderr}')
    return completed


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
if not REPOSITORY_DIR.exists():
    run(['git', 'clone', REPOSITORY_URL, REPOSITORY_DIR])
elif subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPOSITORY_DIR, text=True).strip():
    raise RuntimeError('Repository has local changes; refusing destructive sync.')
run(['git', 'fetch', 'origin', 'main'], cwd=REPOSITORY_DIR)
run(['git', 'checkout', 'main'], cwd=REPOSITORY_DIR)
run(['git', 'merge', '--ff-only', 'origin/main'], cwd=REPOSITORY_DIR)
EXACT_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR, text=True).strip()
ORIGIN_COMMIT = subprocess.check_output(['git', 'rev-parse', 'origin/main'], cwd=REPOSITORY_DIR, text=True).strip()
if EXACT_COMMIT != ORIGIN_COMMIT:
    raise RuntimeError('HEAD is not the exact fetched origin/main commit.')
print('Exact implementation commit:', EXACT_COMMIT)


In [ ]:
run([sys.executable, 'scripts/install_colab_dependencies.py', '--profile', 'training'], cwd=REPOSITORY_DIR)
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-rl.txt'], cwd=REPOSITORY_DIR)
import stable_baselines3
import torch
if stable_baselines3.__version__ != '2.9.0':
    raise RuntimeError(f'Expected SB3 2.9.0, observed {stable_baselines3.__version__}.')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA benchmark requires a Colab GPU runtime.')
print('CUDA device:', torch.cuda.get_device_name(0))


In [ ]:
DATASET_DIR = DRIVE_PROJECT / 'data/modeling/full'
DEMOGRAPHICS_CSV = DRIVE_PROJECT / 'data/raw/clinical.csv'
PROJECT_DATA_ROOT = DRIVE_PROJECT / 'data'
BENCHMARK_DIR = DRIVE_PROJECT / 'outputs/ppo_device_benchmark'
FULL_BASE = DRIVE_PROJECT / 'outputs/ppo_primary_state_full'
required = ['train.npz', 'val.npz', 'test.npz', 'dataset_metadata.json', 'splits']
missing = [name for name in required if not (DATASET_DIR / name).exists()]
if missing or not DEMOGRAPHICS_CSV.is_file():
    raise FileNotFoundError(f'Missing inputs: {missing}; demographics={DEMOGRAPHICS_CSV}')
json.loads((DATASET_DIR / 'dataset_metadata.json').read_text())
print('Dataset paths verified without loading test arrays or outcomes.')
if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))
from src.rl_training.cohort import load_vitaldb_virtual_cohort
preflight_cohort = load_vitaldb_virtual_cohort(DATASET_DIR, demographics_csv=DEMOGRAPHICS_CSV, project_data_root=PROJECT_DATA_ROOT)
COHORT_FINGERPRINT = preflight_cohort.fingerprint
print('Preflight cohort fingerprint:', COHORT_FINGERPRINT)
env = os.environ.copy()
env["VITALDB_MODELING_DATASET_DIR"] = str(DATASET_DIR)
env["VITALDB_DEMOGRAPHICS_CSV"] = str(DEMOGRAPHICS_CSV)
env["VITALDB_PROJECT_DATA_ROOT"] = str(PROJECT_DATA_ROOT)
env["VITALDB_EXPECTED_COHORT_FINGERPRINT"] = COHORT_FINGERPRINT
run([sys.executable, '-m', 'pytest', '-vv', '--tb=short', 'tests/test_ppo_primary_state_full.py'], cwd=REPOSITORY_DIR, env=env, stage='Focused primary-state full pytest', capture_output=True)


In [ ]:
current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_DIR, text=True).strip()
if current_commit != EXACT_COMMIT:
    raise RuntimeError('Repository commit changed after preflight.')
run([sys.executable, 'scripts/benchmark_ppo_primary_state_devices.py', '--device', 'cuda', '--dataset-dir', DATASET_DIR, '--demographics-csv', DEMOGRAPHICS_CSV, '--project-data-root', PROJECT_DATA_ROOT, '--output-dir', BENCHMARK_DIR], cwd=REPOSITORY_DIR)
cuda_summary = json.loads((BENCHMARK_DIR / 'benchmark_summary_cuda.json').read_text())
benchmark_fingerprint = cuda_summary['cohort_fingerprint']
if benchmark_fingerprint != COHORT_FINGERPRINT:
    raise RuntimeError(f'CUDA benchmark cohort mismatch: preflight={COHORT_FINGERPRINT}, benchmark={benchmark_fingerprint}.')
print('CUDA engineering benchmark complete. Scientific full training was not started.')


In [ ]:
cpu_result = BENCHMARK_DIR / 'benchmark_results_cpu.csv'
cuda_result = BENCHMARK_DIR / 'benchmark_results_cuda.csv'
if cpu_result.is_file() and cuda_result.is_file():
    run([sys.executable, 'scripts/benchmark_ppo_primary_state_devices.py', '--output-dir', BENCHMARK_DIR, '--analyze', cpu_result, cuda_result], cwd=REPOSITORY_DIR)
    backend_decision = json.loads((BENCHMARK_DIR / 'analysis/backend_decision.json').read_text())
    if backend_decision['cohort_fingerprint'] != COHORT_FINGERPRINT:
        raise RuntimeError('Merged backend decision cohort differs from the focused-test preflight.')
    print(json.dumps(backend_decision, indent=2))
else:
    backend_decision = None
    print('Place benchmark_results_cpu.csv in the Drive benchmark directory, then rerun this cell.')


In [ ]:
if RUN_FULL_TRAINING:
    if FULL_CONFIRMATION != 'RUN_20_PRIMARY_STATE_FULL_RUNS':
        raise RuntimeError('Exact full-training confirmation is required.')
    if backend_decision is None or backend_decision['selected_backend'] != 'cuda':
        raise RuntimeError('CUDA full training requires a merged benchmark decision selecting CUDA.')
    if backend_decision['cohort_fingerprint'] != COHORT_FINGERPRINT:
        raise RuntimeError('Full runner cohort differs from the focused-test preflight.')
    run([sys.executable, 'scripts/run_ppo_state_full.py', '--device', 'cuda', '--backend-decision', BENCHMARK_DIR / 'analysis/backend_decision.json', '--dataset-dir', DATASET_DIR, '--demographics-csv', DEMOGRAPHICS_CSV, '--project-data-root', PROJECT_DATA_ROOT, '--protocol-dir', FULL_BASE / 'protocol', '--output-root', FULL_BASE / 'runs', '--analysis-dir', FULL_BASE / 'analysis', '--confirmation', FULL_CONFIRMATION], cwd=REPOSITORY_DIR)
else:
    print('Full 20-run training remains locked. RUN_FULL_TRAINING=False.')


In [ ]:
if RUN_ANALYSIS_ONLY:
    protocol_file = FULL_BASE / 'protocol/frozen_primary_state_full_protocol.json'
    if not protocol_file.is_file():
        raise FileNotFoundError('Full protocol has not been frozen yet.')
    frozen = json.loads(protocol_file.read_text())
    run([sys.executable, 'scripts/run_ppo_state_full.py', '--device', frozen['execution_device'], '--dataset-dir', DATASET_DIR, '--demographics-csv', DEMOGRAPHICS_CSV, '--project-data-root', PROJECT_DATA_ROOT, '--protocol-dir', FULL_BASE / 'protocol', '--output-root', FULL_BASE / 'runs', '--analysis-dir', FULL_BASE / 'analysis', '--analysis-only'], cwd=REPOSITORY_DIR)
else:
    print('Analysis-only is disabled until requested.')
